In [18]:
import pandas as pd
import re
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import nltk
from collections import Counter

In [19]:
# Download NLTK data
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\anjit\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\anjit\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\anjit\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [20]:
# Load data
data = pd.read_csv('Data.csv')

# Display the first few rows
print("Data Preview:")
print(data.head())

# Ensure the "Data" column is string type
data["Data"] = data["Data"].astype(str)

Data Preview:
                                                Data
0  Watch or listen live weekdays at 8:30am MT at ...
1  Watch or listen live weekdays at 8:30am MT at ...
2               Chubby And Hot, Always Stir The Pot!
3               Chubby And Hot, Always Stir The Pot!
4  Journalist, publisher of Rebel News — telling ...


In [21]:
# 1. How many records have a date expressed without using alphabets?
date_pattern = r'\b\d{1,4}[-/.]\d{1,2}[-/.]\d{1,4}\b'
date_matches = data["Data"].str.contains(date_pattern, na=False)
date_count = date_matches.sum()
print(f"\n1. Records with dates (no alphabets): {date_count}")


1. Records with dates (no alphabets): 4


In [22]:
# 2. How many records have a word starting with the letter "w"?
w_pattern = r'\bw\w*'
w_matches = data["Data"].str.contains(w_pattern, case=False, na=False)
w_count = w_matches.sum()
print(f"\n2. Records with words starting with 'w': {w_count}")


2. Records with words starting with 'w': 3591


In [23]:
# 3. How many records have a word starting with an alphabet and is not a URL?
word_pattern = r'\b[a-zA-Z]'
url_pattern = r'http[s]?://\S+'
non_url_matches = data["Data"].str.contains(word_pattern, case=False, na=False) & ~data["Data"].str.contains(url_pattern, na=False)
non_url_count = non_url_matches.sum()
print(f"\n3. Records with words starting with an alphabet and not a URL: {non_url_count}")


3. Records with words starting with an alphabet and not a URL: 7301


In [24]:
pattern_word = r'\b[a-zA-Z]'

# Regular expression for matching URLs
pattern_url = r'http[s]?://\S+'

# Check if the "Data" column contains words starting with an alphabet and exclude URLs
matches = data["Data"].str.contains(pattern_word, case=False, na=False) & ~data["Data"].str.contains(pattern_url, na=False)

# Count the number of records that match
count = matches.sum()

print(f"Number of records with a word starting with an alphabet and not a URL: {count}")

Number of records with a word starting with an alphabet and not a URL: 7301


In [25]:
# 4. How many tweets contain one of the emojis :), :D, ;), :P?
emoji_pattern = r'[:;][)DP]'
emoji_matches = data["Data"].str.contains(emoji_pattern, case=False, na=False)
emoji_count = emoji_matches.sum()
print(f"\n4. Records with emojis :), :D, ;), :P: {emoji_count}")


4. Records with emojis :), :D, ;), :P: 18


In [26]:
# 5. How many records contain a decimal number?
decimal_pattern = r'\b\d+\.\d+\b'
decimal_matches = data["Data"].str.contains(decimal_pattern, na=False)
decimal_count = decimal_matches.sum()
print(f"\n5. Records with decimal numbers: {decimal_count}")


5. Records with decimal numbers: 25


In [27]:
# 6. Total number of IP addresses across all records
ip_pattern = r"(\b(?:\d{1,3}\.){3}\d{1,3}\b)"
ip_addresses = data["Data"].str.extractall(ip_pattern)[0].dropna()
ip_count = ip_addresses.size
print(f"\n6. Total number of IP addresses: {ip_count}")
print("Unique IP addresses found:", ip_addresses.unique())


6. Total number of IP addresses: 0
Unique IP addresses found: []


In [28]:
# 7. How many records have a new line?
newline_matches = data["Data"].str.contains(r'\n', na=False)
newline_count = newline_matches.sum()
print(f"\n7. Records with new lines: {newline_count}")


7. Records with new lines: 1211


In [29]:
# 8. Total number of hashtags across all tweets
hashtag_pattern = r'#\w+'
hashtag_matches = data["Data"].str.count(hashtag_pattern)
hashtag_count = hashtag_matches.sum()
print(f"\n8. Total number of hashtags: {hashtag_count}")


8. Total number of hashtags: 2924


In [30]:
# 9. Code to substitute all non-alphanumeric characters with a new line
non_alphanumeric_pattern = r'[^\w\s]'
data["Data_Cleaned"] = data["Data"].str.replace(non_alphanumeric_pattern, '\n', regex=True)
print("\n9. Code to substitute non-alphanumeric characters with a new line applied to 'Data_Cleaned' column.")


9. Code to substitute non-alphanumeric characters with a new line applied to 'Data_Cleaned' column.


In [31]:
# 10. Total number of URLs across all tweets
url_matches = data["Data"].str.count(url_pattern)
url_count = url_matches.sum()
print(f"\n10. Total number of URLs: {url_count}")


10. Total number of URLs: 4


In [34]:
stop_words = set(stopwords.words('english'))
data["Lemmatized_Tokens"] = data["Data"].apply(lambda x: [word for word in word_tokenize(x.lower()) if word.isalnum() and word not in stop_words])

In [35]:
stemmer = PorterStemmer()
data["Tokens"] = data["Lemmatized_Tokens"]  # Ensure 'Tokens' exists for stemming
data["Stemmed_Tokens"] = data["Tokens"].apply(lambda tokens: [stemmer.stem(token) for token in tokens])

In [36]:
# Count unique tokens before and after stemming
unique_tokens_before_stemming = len(set([token for tokens in data["Tokens"] for token in tokens]))
unique_tokens_after_stemming = len(set([token for tokens in data["Stemmed_Tokens"] for token in tokens]))
print(f"\nUnique tokens before stemming: {unique_tokens_before_stemming}")
print(f"Unique tokens after stemming: {unique_tokens_after_stemming}")


Unique tokens before stemming: 7214
Unique tokens after stemming: 5927


In [37]:
from collections import Counter

stemmed_counts = Counter([token for tokens in data["Stemmed_Tokens"] for token in tokens])
lemmatized_counts = Counter([token for tokens in data["Lemmatized_Tokens"] for token in tokens])

top_10_stemmed = stemmed_counts.most_common(10)
top_10_lemmatized = lemmatized_counts.most_common(10)

print("\nTop 10 words after stemming:", top_10_stemmed)
print("Top 10 words after lemmatization:", top_10_lemmatized)



Top 10 words after stemming: [('alberta', 1061), ('outdoor', 682), ('news', 604), ('commun', 595), ('canada', 559), ('adventur', 530), ('travel', 459), ('follow', 446), ('love', 410), ('calgari', 390)]
Top 10 words after lemmatization: [('alberta', 1059), ('news', 604), ('canada', 559), ('outdoor', 549), ('calgary', 390), ('writer', 383), ('community', 379), ('adventures', 373), ('follow', 361), ('photographer', 359)]
